“The objective of this project is to analyse a retail bank's customer portfolio, accounts, lending risk, transaction behaviour and branch performance, and turn the data into business insights and recommendations.”

In [ ]:
# import libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [ ]:
# Load data 

accounts = pd.read_csv('../data/accounts.csv')
branches = pd.read_csv('../data/branches.csv')
customers = pd.read_csv('../data/customers.csv')
loans = pd.read_csv('../data/loans.csv')
transactions = pd.read_csv('../data/transactions.csv')

In [ ]:
# Put datasets together

datasets = {
 'accounts': accounts,
 'branches': branches,
 'customers': customers,
 'loans': loans,
 'transactions': transactions

}

# Now inspect them using loop

for name, df in datasets.items():
    print(f'\n --- {name.upper()}--')
    print('Shape :', df.shape)
    print('columns:',df.columns.tolist())
    print('Missing Values',df.isnull().sum())
    print(' Duplicates:',df.duplicated().sum())



In [ ]:
summary = []

for name, df in datasets.items():
    summary.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Missing Values': df.isna().sum().sum(),
        'Duplicates': df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

In [ ]:
customers.isna().sum()

In [ ]:
customers[customers['Geography'].isna()]

In [ ]:
customers[customers['EstimatedAnnualIncome'].isna()]

In [ ]:
customers.groupby('Geography')['EstimatedAnnualIncome'].mean()

In [ ]:
customers.groupby('Geography')['EstimatedAnnualIncome'].median()

In [ ]:
# MEDIAN is giving us better value as mean giving us annual income slightly higher
# # Now fill the missing values using median and transform which fill only missing column with median income  

customers['EstimatedAnnualIncome'] = customers['EstimatedAnnualIncome'].fillna(customers.groupby('Geography')['EstimatedAnnualIncome'].transform('median'))

In [ ]:
customers.isna().sum()

# We can't invent geogrpahy so lets fill with unknown 

customers['Geography'] = customers['Geography'].fillna('Unknown')

## we will not fill branch id with missing values only change data type 

customers['BranchID'] = customers['BranchID'].astype('Int64')

In [ ]:
# Lets answers business questions now 

# How many customers are in each country?
customers['Geography'].value_counts()

#What is average income by country
customers.groupby('Geography')['EstimatedAnnualIncome'].mean()

#What is average credit score by country
customers.groupby('Geography')['CreditScore'].mean()


#How many customers are active vs Inactive
customers['IsActiveMember'].value_counts()


### Account / Depost Questions 

In [ ]:
# how many accounts of each type
accounts['AccountType'].value_counts()

## total customer balance by account type
accounts.groupby('AccountType')['CurrentBalance'].sum()

## Average balance by account type
accounts.groupby('AccountType')['CurrentBalance'].mean()


## Open vs Dormant account
accounts['AccountStatus'].value_counts()


## Lending / Risk Questions 

In [ ]:
# No of loan by type 
loans['LoanType'].value_counts()

# Money currently exposed by lOAN TYPE 
loans.groupby('LoanType')['OutstandingBalance'].sum()

# Current/Late / Default
loans['LoanStatus'].value_counts(normalize=True)*100

# Which loan type has the highest  default rate
pd.crosstab(loans['LoanType'],loans['LoanStatus'],normalize='index')*100

## Transaction Questions 

In [ ]:
transactions.groupby('Month')['TransactionCount'].sum()

transactions.groupby('Month')['InflowAmount'].sum()

transactions.groupby('Month')['OutflowAmount'].sum()

transactions.groupby('Month')['FeeRevenue'].sum()

transactions.groupby('Month')['DigitalTransactionShare'].mean()

Now we merge table to answer questions 

In [ ]:
# How many transactions are happening at frankfurt central
 # to answer these question we will merge tables some of them have matching some don'ts

#lets merge transactions with account 

accounts_transactions = pd.merge(transactions, accounts, on='AccountID', how='left')
accounts_transactions


In [ ]:
accounts_transactions.columns.tolist()

In [ ]:
(accounts_transactions['CustomerID_x'] == accounts_transactions['CustomerID_y']).all()

In [ ]:
# when we merge transactions with account both have customerid we used accountId to merge them so created two columns customeridx and customery we are going
# clean them


# we delete cusomterID_y  column first 
accounts_transactions = accounts_transactions.drop(columns='CustomerID_y')


In [ ]:

# We rename the CustomerID_x column to columnID 
accounts_transactions = accounts_transactions.rename(
    columns={'CustomerID_x':'CustomerID'}
)

In [ ]:
# Now merge this with customer 

bank_data = pd.merge(
    accounts_transactions,
    customers,
    on='CustomerID',
    how='left'
)

In [ ]:
bank_data

In [ ]:
# Now merge bank data with branches on branchID 

bank_data = pd.merge(
    bank_data,
    branches,
    on='BranchID',
    how='left'
)

In [ ]:
bank_data[['CustomerID','BranchID','BranchName']].head()

In [ ]:
loan_summary = loans.groupby('CustomerID').agg(
    TotalLoanOutstanding=('OutstandingBalance', 'sum'),
    NumberOfLoans=('LoanID', 'count')
).reset_index()

In [ ]:
bank_data = pd.merge(
    bank_data,
    loan_summary,
    on='CustomerID',
    how='left'
)

In [ ]:
bank_data['TotalLoanOutstanding'] = bank_data['TotalLoanOutstanding'].fillna(0)
bank_data['NumberOfLoans'] = bank_data['NumberOfLoans'].fillna(0)

In [ ]:
bank_data[['CustomerID', 'TotalLoanOutstanding', 'NumberOfLoans']].head()

In [ ]:
# ==========================================
# CROSS-TABLE BUSINESS ANALYSIS
# ==========================================

# Q1: Which branch has the most customers?
print("\nQ1 - Customers by Branch")
print(
    bank_data.groupby('BranchName')['CustomerID']
    .nunique()
    .sort_values(ascending=False)
)


# Q2: Which branch manages the most accounts?
print("\nQ2 - Accounts by Branch")
print(
    bank_data.groupby('BranchName')['AccountID']
    .nunique()
    .sort_values(ascending=False)
)


# Q3: Which branch generates the most transaction activity?
print("\nQ3 - Transaction Volume by Branch")
print(
    bank_data.groupby('BranchName')['TransactionCount']
    .sum()
    .sort_values(ascending=False)
)


# Q4: Which branch receives the highest total inflows?
print("\nQ4 - Inflows by Branch")
print(
    bank_data.groupby('BranchName')['InflowAmount']
    .sum()
    .sort_values(ascending=False)
)


# Q5: Which branch generates the most fee revenue?
print("\nQ5 - Fee Revenue by Branch")
print(
    bank_data.groupby('BranchName')['FeeRevenue']
    .sum()
    .sort_values(ascending=False)
)


# Q6: Which branch has the highest digital transaction share?
print("\nQ6 - Digital Share by Branch")
print(
    (bank_data.groupby('BranchName')['DigitalTransactionShare']
     .mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)


# Q7: Which geography has the highest average customer credit score?
print("\nQ7 - Average Credit Score by Geography")
print(
    bank_data.groupby('Geography')['CreditScore']
    .mean()
    .sort_values(ascending=False)
)


# Q8: Which customer segment has the highest average loan outstanding?
print("\nQ8 - Average Loan Outstanding by Geography")
print(
    bank_data.groupby('Geography')['TotalLoanOutstanding']
    .mean()
    .sort_values(ascending=False)
)

## Data Visualization 

In [ ]:
#Customers by country - Bar chart

customers['Geography'].value_counts().plot(kind='bar')

plt.title('Customers by Country')
plt.xlabel('Country')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# total balance by acount type
accounts.groupby('AccountType')['CurrentBalance'].sum() \
    .sort_values(ascending=False).plot(kind='bar')

plt.title('Total Balance by Account Type')
plt.xlabel('Account Type')
plt.ylabel('Total Balance (£)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Loan status by loan type - stacked bar

loan_status = pd.crosstab(
    loans['LoanType'],
    loans['LoanStatus'],
    normalize='index'
) * 100

loan_status.plot(kind='bar', stacked=True)

plt.title('Loan Status by Loan Type')
plt.xlabel('Loan Type')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')
plt.show()

In [ ]:
# Monthly transaction volume - line chart

monthly_transactions = (
    transactions.groupby('Month')['TransactionCount'].sum()
)

monthly_transactions.plot(
    kind='line',
    marker='o'
)

plt.title('Monthly Transaction Volume')
plt.xlabel('Month')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Monthly inflows vs outflows — Line chart

monthly_flows = (
    transactions.groupby('Month')[['InflowAmount', 'OutflowAmount']]
    .sum()
)

monthly_flows.plot(
    kind='line',
    marker='o'
)

plt.title('Monthly Inflows vs Outflows')
plt.xlabel('Month')
plt.ylabel('Amount (£)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Transaction volume by branch - bar chart

branch_transactions = (
    bank_data.groupby('BranchName')['TransactionCount']
    .sum()
    .sort_values(ascending=False)
)

branch_transactions.plot(kind='bar')

plt.title('Transaction Volume by Branch')
plt.xlabel('Branch')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.show()

# Key Insights

### Customer Portfolio
- The UK is the bank's largest customer market, with 996 customers.
- Customer distribution varies significantly across the four European markets.

### Deposits and Accounts
- Current accounts are the most widely held account type.
- Savings accounts hold the largest total customer balance.
- ISA accounts have the highest average balance despite having fewer accounts.

### Lending and Credit Risk
- Mortgages represent the largest share of outstanding lending exposure.
- Personal loans have the highest default rate at approximately 3.73%.
- Auto loans have the highest late-payment rate at approximately 10.13%.
- Mortgage loans have the strongest repayment performance, with over 91% classified as current.

### Transaction Behaviour
- Transaction volume remains relatively stable throughout the year at around 50,000 transactions per month.
- Total annual inflows (£129.9m) exceed total outflows (£101.3m), indicating positive net customer cash flow.
- Approximately 67% of transaction activity is digital.

### Branch Performance
- London Central is the largest branch by customers, accounts and transaction volume.
- Paris Central is the second-largest branch by transaction activity.
- Digital transaction share is relatively consistent across branches, at approximately 67%.

# Business Recommendations

1. **Monitor Personal Loan Credit Risk**
   - Personal loans have the highest default rate. The bank should review underwriting criteria and identify higher-risk customer segments.

2. **Investigate Auto Loan Delinquencies**
   - Auto loans show the highest late-payment rate. Early-warning monitoring and proactive customer contact could reduce future defaults.

3. **Protect and Grow Savings Deposits**
   - Savings accounts hold the largest total balance. Retention campaigns and competitive savings products could help protect this important deposit base.

4. **Increase Digital Adoption**
   - Digital transactions already represent roughly 67% of activity. The bank could target customers with lower digital usage to reduce servicing costs and improve digital engagement.

5. **Use London as a Performance Benchmark**
   - London Central leads in customers, accounts and transaction activity. Management should investigate whether successful practices can be replicated across lower-volume branches.